# 03 — Fine-tuning DistilBERT (Google Colab GPU)

**Projet** : FakeNews Analyzer — DevComplex  
**⚠️ À exécuter sur Google Colab avec GPU T4 (Runtime → Modifier le type de runtime → T4 GPU)**

**Prérequis** : Uploader `09_data/colab_exports/` sur Google Drive avant d'exécuter.

---

In [ ]:
# ============================================================
# Fichier  : 03_distilbert_finetune.ipynb
# Rôle     : Fine-tuning DistilBERT pour la classification FAKE/REAL
# Version  : V1 (Colab GPU)
# Projet   : FakeNews Analyzer — DevComplex
# Auteur   : DevComplex
# ============================================================
# Exécuter sur Google Colab avec GPU T4
# ============================================================

import os
import json
import time
import pandas as pd
import numpy as np

# Monter Google Drive
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR  = '/content/drive/MyDrive/fakenews_colab'
OUTPUT_DIR = '/content/distilbert_output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('✓ Google Drive monté')
print(f'Répertoire données : {DRIVE_DIR}')

In [ ]:
# Installer les dépendances supplémentaires si nécessaire
# !pip install transformers datasets accelerate -q

import torch
from transformers import (
    DistilBertForSequenceClassification,
    DistilBertTokenizerFast,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU : {torch.cuda.get_device_name(0)}')
    print(f'VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## Section 1 — Chargement des données

In [ ]:
MODEL_NAME = 'distilbert-base-uncased'
MAX_LENGTH = 256

train_df = pd.read_csv(os.path.join(DRIVE_DIR, 'train_texts.csv'))
test_df  = pd.read_csv(os.path.join(DRIVE_DIR, 'test_texts.csv'))

train_df = train_df.dropna(subset=['clean_text'])
test_df  = test_df.dropna(subset=['clean_text'])

print(f'Train : {len(train_df):,} | Test : {len(test_df):,}')
print(f'Distribution train — FAKE: {train_df["label"].sum():,} | REAL: {(train_df["label"]==0).sum():,}')

## Section 2 — Tokenisation

In [ ]:
tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    return tokenizer(
        examples['clean_text'],
        truncation=True,
        max_length=MAX_LENGTH,
        padding='max_length'
    )

train_dataset = Dataset.from_pandas(train_df[['clean_text', 'label']])
test_dataset  = Dataset.from_pandas(test_df[['clean_text', 'label']])

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset  = test_dataset.map(tokenize_function, batched=True)

train_dataset = train_dataset.rename_column('label', 'labels')
test_dataset  = test_dataset.rename_column('label', 'labels')
train_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
test_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])

print(f'✓ Tokenisation terminée')

## Section 3 — Entraînement

In [ ]:
model = DistilBertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={0: 'REAL', 1: 'FAKE'},
    label2id={'REAL': 0, 'FAKE': 1}
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    probas = torch.softmax(torch.tensor(logits), dim=-1).numpy()[:, 1]
    return {
        'accuracy': accuracy_score(labels, predictions),
        'f1': f1_score(labels, predictions, average='weighted'),
        'auc': roc_auc_score(labels, probas),
    }

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    weight_decay=0.01,
    fp16=True,                          # Mixed precision (GPU uniquement)
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    logging_steps=100,
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print('Démarrage de l\'entraînement...')
trainer.train()

## Section 4 — Évaluation finale et sauvegarde

In [ ]:
results = trainer.evaluate()
print('\n=== RÉSULTATS FINAUX DistilBERT ===')
for k, v in results.items():
    print(f'  {k} : {v:.4f}' if isinstance(v, float) else f'  {k} : {v}')

In [ ]:
# Sauvegarder le modèle et le tokenizer
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'✓ Modèle sauvegardé dans {OUTPUT_DIR}')

# Compresser et télécharger
import shutil
zip_path = '/content/distilbert_model.zip'
shutil.make_archive('/content/distilbert_model', 'zip', OUTPUT_DIR)
print(f'✓ Archive créée : {zip_path}')

from google.colab import files
files.download(zip_path)
print('✓ Téléchargement lancé — dézipper dans 04_models/distilbert/')